# Notebook 1b — Concept Vocabulary

Splits all SNOMED breast concept preferred terms on spaces and builds a deduplicated word-level dictionary.

**Input**
- `data/embeddings-concept-openai/concepts.csv` — breast concept preferred terms

**Outputs**
- `data/stage1/concept_vocabulary.pkl` — `{word: [concept_id, ...]}` bare words
- `data/stage1/concept_vocabulary_spaced.pkl` — `{" " + word: [concept_id, ...]}` space-prefixed words

In BPE tokenizers (Llama-3, GPT-2 family), a word mid-sentence is encoded with the leading space baked into the token — `"breast"` and `" breast"` have different token IDs. The spaced form is what appears inside phrases and is what the footprints probes need to look up the correct token ID.

**Motivation**

SNOMED concepts are multi-word phrases, not single multi-token words. The footprints probes (notebook 1) detect subword-token merging within a *single word* — they cannot be applied meaningfully to a full phrase like "invasive ductal carcinoma of breast". This notebook extracts the 1,490 individual words across all 1,879 concept preferred terms so that downstream calibration can operate on words rather than phrases.

In [ ]:
# parameters
DATA_DIR     = "../../data/stage1"
CONCEPTS_CSV = "../../data/embeddings-concept-openai/concepts.csv"

In [ ]:
import os
import pickle

import pandas as pd

os.makedirs(DATA_DIR, exist_ok=True)

df = pd.read_csv(CONCEPTS_CSV)
print(f"Concepts loaded: {len(df)}")

In [ ]:
# Build vocabulary: word -> sorted list of concept_ids that contain it
vocab = {}
for _, row in df.iterrows():
    cid = str(row["concept_id"])
    for word in str(row["preferred_term"]).split(" "):
        word = word.strip().strip("()")
        if word:
            if word not in vocab:
                vocab[word] = []
            if cid not in vocab[word]:
                vocab[word].append(cid)

for word in vocab:
    vocab[word].sort()

print(f"Unique words: {len(vocab)}")
print(f"Sample entries:")
for word in list(sorted(vocab))[:5]:
    print(f"  {word!r}: {len(vocab[word])} concept(s)")

In [ ]:
out_path = os.path.join(DATA_DIR, "concept_vocabulary.pkl")
with open(out_path, "wb") as f:
    pickle.dump(vocab, f)

print(f"Saved: {out_path}")

In [ ]:
vocab_spaced = {" " + word: cids for word, cids in vocab.items()}

out_path_spaced = os.path.join(DATA_DIR, "concept_vocabulary_spaced.pkl")
with open(out_path_spaced, "wb") as f:
    pickle.dump(vocab_spaced, f)

print(f"Saved: {out_path_spaced}")